# M6 W6 v2 full retrieval comparison — original and paraphrased queries

This diagnostic notebook compares nine pinned embedding models and six Hybrid
weights on 273 development questions using the repository's shared retrieval
implementation. It does **not** run an all-model sweep on the 68-question
holdout. That holdout was exposed by a superseded run and is retained only for
a two-configuration diagnostic comparison; it is not a clean final Test set.
The joint Dev metric winner is Qwen3 Dense 0.10 / BM25 0.90. BGE-M3 Dense
0.10 / BM25 0.90 remains the team-selected system retriever for the M7 handoff;
it is one combined Complete Hit@5 behind Qwen3. This operational decision is
not a claim of metric or statistical superiority.

The notebook must be executed from a clean committed worktree. Large indexes,
per-question metrics and candidate hits remain outside Git.


## 1. Configuration and reproducibility


In [1]:
from __future__ import annotations

import collections
import gc
import hashlib
import json
import math
import os
import re
import sys
import zipfile
from datetime import UTC, datetime
from pathlib import Path

import numpy as np
import pandas as pd


def write_utf8_lf(path: Path, text: str) -> None:
    path.write_bytes(text.replace("\r\n", "\n").encode("utf-8"))


def write_csv_lf(frame: pd.DataFrame, path: Path) -> None:
    write_utf8_lf(path, frame.to_csv(index=False, lineterminator="\n"))


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "src" / "cs30").is_dir():
            return candidate
    raise FileNotFoundError("Run this notebook from the cs30-personalised-rag checkout.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from cs30.config import AppConfig, RetrievalConfig  # noqa: E402
from cs30.contracts import RetrievalMode, RetrievalResult  # noqa: E402
from cs30.evaluation.manifest import capture_git_state  # noqa: E402
from cs30.pipeline import build_real_retrieval_deps  # noqa: E402
from cs30.retrieval.v2_adapter import materialise_v2_retrieval_view  # noqa: E402

INPUT_DIR = Path(
    os.getenv("CS30_M6_INPUT_DIR", PROJECT_ROOT / "eval_inputs" / "m6_v2")
).expanduser()


def resolve_input(env_name: str, filename: str) -> Path:
    configured = os.getenv(env_name)
    candidate = Path(configured).expanduser() if configured else INPUT_DIR / filename
    if not candidate.is_file():
        raise FileNotFoundError(
            f"Missing {filename}. Set {env_name} or place it under {INPUT_DIR}. "
            "See eval_inputs/m6_v2/README.md."
        )
    return candidate.resolve()


CORPUS_ZIP = resolve_input("CS30_M6_CORPUS_ZIP", "M4_to_M5_four_book_corpus_a1fe99e.zip")
M5_ZIP = resolve_input("CS30_M6_INDEX_ZIP", "M5_to_M6_four_book_2026-10-21.zip")
QUERY_SET = os.getenv("CS30_M6_QUERY_SET", "original").strip().lower()
assert QUERY_SET in {"original", "para_v1"}, QUERY_SET
ORIGINAL_GOLD = resolve_input("CS30_M6_ORIGINAL_GOLD", "gold_v0_1_3.jsonl")
active_gold_filename = (
    "gold_v0_1_3.jsonl" if QUERY_SET == "original" else "gold_v0_1_3_para_v1.jsonl"
)
INTEGRATED_GOLD = resolve_input("CS30_M6_GOLD", active_gold_filename)
GITHUB_GOLD = PROJECT_ROOT / "m3_gold" / "gold_v0_1_1.jsonl"

WORK_DIR = Path(
    os.getenv("CS30_M6_WORK_DIR", PROJECT_ROOT / "artifacts" / "m6_w6_retrieval_four_book")
).expanduser()
OUTPUT_SLUG = (
    "outputs_four_book_gold_v0_1_3_joint_rule_original"
    if QUERY_SET == "original"
    else "outputs_four_book_gold_v0_1_3_joint_rule_para_v1"
)
ORIGINAL_OUTPUT_DIR = WORK_DIR / "outputs_four_book_gold_v0_1_3_joint_rule_original"
PARA_OUTPUT_DIR = WORK_DIR / "outputs_four_book_gold_v0_1_3_joint_rule_para_v1"
OUTPUT_DIR = Path(
    os.getenv(
        "CS30_M6_OUTPUT_DIR",
        WORK_DIR / OUTPUT_SLUG,
    )
).expanduser()
PUBLISHED_RESULT_SLUG = (
    "m6_w6_v2_four_book_gold_v0_1_3"
    if QUERY_SET == "original"
    else "m6_w6_v2_four_book_gold_v0_1_3_para_v1"
)
PUBLISHED_RESULT_DIR = PROJECT_ROOT / "docs" / "results" / PUBLISHED_RESULT_SLUG
INDEX_DIR = WORK_DIR / "indexes_four_book"
MODEL_CACHE = Path(os.getenv("HF_HOME", Path.home() / ".cache" / "huggingface")).expanduser()

RUN_EXPENSIVE = os.getenv("CS30_M6_RUN_MODELS", "0") == "1"
FORMAL_MODE = os.getenv("CS30_M6_FORMAL_MODE", "0") == "1"
DEVICE = os.getenv("CS30_M6_DEVICE", "cuda")
SYSTEM_RETRIEVER_MODEL = "bge_m3"
SYSTEM_RETRIEVER_MODE = "hybrid_d0.10_b0.90"
SYSTEM_SELECTION_BASIS = (
    "team system decision after joint original/para-v1 Dev review on 2026-10-23; "
    "BGE-M3 is retained for the downstream M7 handoff and is not described as "
    "the joint metric winner"
)
SYSTEM_MAX_COMPLETE_HIT_GAP = 1

TOP_K = (1, 3, 5, 10)
FINAL_TOP_K = max(TOP_K)
RRF_K = 60
RRF_INPUT_TOP_K = 20
HYBRID_WEIGHTS = (
    (0.95, 0.05),
    (0.90, 0.10),
    (0.80, 0.20),
    (0.75, 0.25),
    (0.50, 0.50),
    (0.25, 0.75),
)  # (BM25, Dense)

GOLD_ACCEPTANCE_POLICY = "user_accepted_for_m6_evaluation"
EXPECTED_CORPUS_ZIP_SHA256 = (
    "sha256:074ff89686fe3649089329f7858505f7a68d228d0284a83394a0c0b5a547a592"
)
EXPECTED_M5_ZIP_SHA256 = "sha256:2851abf71164e131b9064de0e87385b3b7f38943cbfcbf420dad5a40c0bf8c3e"
EXPECTED_ORIGINAL_GOLD_SHA256 = (
    "sha256:7d69af416591b64198bf25806424347366eb2123fdcdcf36ad757d37c4f942b8"
)
EXPECTED_GOLD_SHA256 = {
    "original": EXPECTED_ORIGINAL_GOLD_SHA256,
    "para_v1": "sha256:57ca94918912df01d8d66e6e9e7fed95b3faf4e6dd4254428b349823a3623696",
}[QUERY_SET]
EXPECTED_GOLD_SOURCE_CORPUS_VERSIONS = frozenset(
    {"2.0.0-dev.m4-structure-v2-main-3553825", "ck12-people-physics-2016-09-09"}
)
EXPECTED_GOLD_ANNOTATION_VERSION = {
    "original": "m3_gold_v0.1.3",
    "para_v1": "m3_gold_v0.1.3-para-v1",
}[QUERY_SET]
EXPECTED_GITHUB_GOLD_SHA256 = (
    "sha256:52d8bab9655ae2f4df2a8bf883a7cbc87acbe83dbb11af983bc228c1412839ad"
)
FROZEN_EVALUATION_SET_VERSION = None
FROZEN_GOLD_ANNOTATION_VERSION = None

for path in (PROJECT_ROOT, CORPUS_ZIP, M5_ZIP, ORIGINAL_GOLD, INTEGRATED_GOLD, GITHUB_GOLD):
    assert path.exists(), f"Missing input: {path}"

print("Repository:", PROJECT_ROOT)
print("Query set:", QUERY_SET)
print("Input source: environment overrides or", INPUT_DIR)
print("Working output:", OUTPUT_DIR)
print("Published results:", PUBLISHED_RESULT_DIR.relative_to(PROJECT_ROOT))
print("Expensive run enabled:", RUN_EXPENSIVE, "device:", DEVICE)

Repository: E:\Sydney\vscode\w5_20_m6_golden_evidence\.codex_worktrees\display_full_summary
Input source: environment overrides or E:\Sydney\vscode\w5_20_m6_golden_evidence\.codex_worktrees\display_full_summary\eval_inputs\m6_v2
Working output: E:\Sydney\vscode\w5_20_m6_golden_evidence\.codex_worktrees\display_full_summary\artifacts\m6_w6_retrieval_four_book\outputs_four_book_gold_v0_1_3_shared_retriever_273_68
Published results: docs\results\m6_w6_v2_four_book_gold_v0_1_3
Expensive run enabled: False device: cuda


## 2. Pinned model registry and four-book inputs


In [2]:
MODEL_SPECS = {
    "minilm": {
        "model": "sentence-transformers/all-MiniLM-L6-v2",
        "revision": "1110a243fdf4706b3f48f1d95db1a4f5529b4d41",
        "dimension": 384,
        "batch": 64,
    },
    "mpnet": {
        "model": "sentence-transformers/all-mpnet-base-v2",
        "revision": "e8c3b32edf5434bc2275fc9bab85f82640a19130",
        "dimension": 768,
        "batch": 32,
    },
    "e5_base": {
        "model": "intfloat/e5-base-v2",
        "revision": "f52bf8ec8c7124536f0efb74aca902b2995e5bcd",
        "dimension": 768,
        "batch": 32,
        "query_instruction": "query: ",
        "document_prefix": "passage: ",
    },
    "bge_base": {
        "model": "BAAI/bge-base-en-v1.5",
        "revision": "a5beb1e3e68b9ab74eb54cfd186867f64f240e1a",
        "dimension": 768,
        "batch": 32,
        "query_instruction": "Represent this sentence for searching relevant passages: ",
    },
    "jina": {
        "model": "jinaai/jina-embeddings-v2-small-en",
        "revision": "44e7d1d6caec8c883c2d4b207588504d519788d0",
        "dimension": 512,
        "batch": 16,
        "trust_remote_code": True,
    },
    "gte_modernbert": {
        "model": "Alibaba-NLP/gte-modernbert-base",
        "revision": "e7f32e3c00f91d699e8c43b53106206bcc72bb22",
        "dimension": 768,
        "batch": 4,
        "trust_remote_code": True,
        "m5_path": "gte-modernbert",
    },
    "nomic": {
        "model": "nomic-ai/nomic-embed-text-v1.5",
        "revision": "e9b6763023c676ca8431644204f50c2b100d9aab",
        "dimension": 768,
        "batch": 8,
        "trust_remote_code": True,
        "query_instruction": "search_query: ",
        "document_prefix": "search_document: ",
    },
    "bge_m3": {
        "model": "BAAI/bge-m3",
        "revision": "5617a9f61b028005a4858fdac845db406aefb181",
        "dimension": 1024,
        "batch": 4,
        "m5_path": "bge-m3",
    },
    "qwen3": {
        "model": "Qwen/Qwen3-Embedding-0.6B",
        "revision": "97b0c614be4d77ee51c0cef4e5f07c00f9eb65b3",
        "dimension": 1024,
        "batch": 2,
        "m5_path": "qwen3-embedding",
        "query_prompt_name": "query",
    },
}
for spec in MODEL_SPECS.values():
    spec.setdefault("query_instruction", "")
    spec.setdefault("query_prompt_name", None)
    spec.setdefault("document_prefix", "")
    spec.setdefault("trust_remote_code", False)
    assert re.fullmatch(r"[0-9a-f]{40}", spec["revision"])


def sha256(data: bytes) -> str:
    return "sha256:" + hashlib.sha256(data).hexdigest()


def read_jsonl_bytes(data: bytes) -> list[dict]:
    return [json.loads(line) for line in data.decode("utf-8-sig").splitlines() if line.strip()]


with zipfile.ZipFile(CORPUS_ZIP) as archive:
    prefixes = {
        name.removesuffix("records.jsonl")
        for name in archive.namelist()
        if name.endswith("records.jsonl")
    }
    assert len(prefixes) == 1, sorted(prefixes)
    CORPUS_PREFIX = next(iter(prefixes))
    corpus_bytes = archive.read(CORPUS_PREFIX + "records.jsonl")
    manifest_bytes = archive.read(CORPUS_PREFIX + "manifest.json")
    corpus_manifest = json.loads(manifest_bytes)
    chunks = read_jsonl_bytes(corpus_bytes)
    delivered_checksums = archive.read(CORPUS_PREFIX + "SHA256SUMS").decode("utf-8")
    for line in delivered_checksums.splitlines():
        expected, filename = line.split(None, 1)
        actual = hashlib.sha256(archive.read(CORPUS_PREFIX + filename.strip())).hexdigest()
        assert actual == expected, filename

gold_bytes = INTEGRATED_GOLD.read_bytes()
gold = read_jsonl_bytes(gold_bytes)
original_gold_bytes = ORIGINAL_GOLD.read_bytes()
original_gold = read_jsonl_bytes(original_gold_bytes)
github_gold_bytes = GITHUB_GOLD.read_bytes()
github_gold = read_jsonl_bytes(github_gold_bytes)
ordered_chunk_ids = [row["chunk_id"] for row in chunks]
by_chunk = {row["chunk_id"]: row for row in chunks}

assert sha256(CORPUS_ZIP.read_bytes()) == EXPECTED_CORPUS_ZIP_SHA256
assert sha256(M5_ZIP.read_bytes()) == EXPECTED_M5_ZIP_SHA256
assert sha256(original_gold_bytes) == EXPECTED_ORIGINAL_GOLD_SHA256
assert sha256(gold_bytes) == EXPECTED_GOLD_SHA256
assert sha256(github_gold_bytes) == EXPECTED_GITHUB_GOLD_SHA256
assert sha256(corpus_bytes) == corpus_manifest["corpus_hash"]
assert len(chunks) == corpus_manifest["record_count"] == 6184
assert len(ordered_chunk_ids) == len(set(ordered_chunk_ids))
assert len(gold) == 341 and len({row["question_id"] for row in gold}) == 341
assert len(original_gold) == 341
original_by_id = {row["question_id"]: row for row in original_gold}
gold_by_id = {row["question_id"]: row for row in gold}
assert original_by_id.keys() == gold_by_id.keys()
if QUERY_SET == "para_v1":
    invariant_fields = set(next(iter(original_by_id.values()))) - {
        "question",
        "gold_annotation_version",
    }
    assert all(
        all(
            original_by_id[question_id][field] == gold_by_id[question_id][field]
            for field in invariant_fields
        )
        for question_id in original_by_id
    )
    assert (
        sum(
            original_by_id[question_id]["question"] != gold_by_id[question_id]["question"]
            for question_id in original_by_id
        )
        == 336
    )
assert len(github_gold) == 20
assert all(row["answerable"] is True for row in gold)
gold_source_corpus_versions = {row["corpus_version"] for row in gold}
assert gold_source_corpus_versions == EXPECTED_GOLD_SOURCE_CORPUS_VERSIONS
assert {row["gold_annotation_version"] for row in gold} == {EXPECTED_GOLD_ANNOTATION_VERSION}
assert {row["annotation_status"] for row in gold} == {"draft"}
assert corpus_manifest["reportable"] is False and corpus_manifest["mode"] == "development"

print("Pinned models:", len(MODEL_SPECS))
print("Corpus:", corpus_manifest["corpus_version"], "chunks:", len(chunks))
print("Working Gold rows:", len(gold), "annotation status: draft")
print("Gold SHA-256:", sha256(gold_bytes))

Pinned models: 9
Corpus: 2.0.0-dev.m4-four-book-main-a1fe99e chunks: 6184
Working Gold rows: 341 annotation status: draft


## 3. Bind every Gold evidence span to the four-book corpus


In [3]:
def normalized(text: str) -> str:
    return " ".join(text.casefold().split())


block_to_chunks = collections.defaultdict(set)
by_chunk = {c["chunk_id"]: c for c in chunks}
for chunk in chunks:
    for span in chunk["spans"]:
        block_to_chunks[span["block_id"]].add(chunk["chunk_id"])

binding_rows = []
gold_paths = {}
for question in gold:
    qid = question["question_id"]
    paths = []
    for group in question["gold_core_evidence_sets"]:
        mapped = []
        for span in group:
            candidates = block_to_chunks.get(span.get("block_id"), set())
            assert len(candidates) == 1, (qid, span.get("span_id"), "non-unique/missing block")
            chunk_id = next(iter(candidates))
            chunk = by_chunk[chunk_id]
            assert str(chunk["chapter_id"]) == str(span["chapter_id"]), (qid, "chapter mismatch")
            assert normalized(span["verbatim_text"]) in normalized(chunk["text"]), (
                qid,
                "verbatim mismatch",
            )
            mapped.append(chunk_id)
            binding_rows.append(
                {
                    "question_id": qid,
                    "span_id": span["span_id"],
                    "block_id": span["block_id"],
                    "chunk_id": chunk_id,
                    "textbook_id": chunk["textbook_id"],
                    "method": "unique_block_id+chapter+verbatim",
                }
            )
        paths.append(frozenset(mapped))
    assert paths, (qid, "no core evidence")
    gold_paths[qid] = paths
assert len(binding_rows) == 352
assert len(gold_paths) == 341
binding_bytes = (
    "\n".join(json.dumps(r, sort_keys=True, ensure_ascii=False) for r in binding_rows) + "\n"
).encode("utf-8")
print(
    "Bound spans:",
    len(binding_rows),
    "of",
    sum(len(p) for q in gold for p in q["gold_core_evidence_sets"]),
)
print("Binding SHA-256:", sha256(binding_bytes))

Bound spans: 352 of 352
Binding SHA-256: sha256:99b72302fd335676f5ff8b53516972db361e81eca97aa52b0ee050086b4c055e


## 4. Formal gate and diagnostic 273/68 split


In [4]:
EVALUATION_SET_VERSION = (
    f"m6-v2-diagnostic-gold-341-v0.1.3-{QUERY_SET}-split-273-68-exposed-2026-10-23"
)
GOLD_ANNOTATION_VERSION = sorted({g["gold_annotation_version"] for g in gold})


def assert_formal_ready() -> None:
    assert corpus_manifest["reportable"] is True, "M4 corpus is development-only."
    assert corpus_manifest["mode"] != "development"
    assert "ck12" in " ".join(corpus_manifest["included_textbook_ids"]).lower(), (
        "CK-12 source absent."
    )
    assert GOLD_ACCEPTANCE_POLICY == "user_accepted_for_m6_evaluation"
    assert {g["split"] for g in gold} <= {"dev", "test"}, "Dev/Test splits are not frozen."
    assert all(g["corpus_version"] == corpus_manifest["corpus_version"] for g in gold), (
        "Formal Gold must be native to the frozen corpus."
    )
    assert FROZEN_EVALUATION_SET_VERSION, "Missing approved evaluation_set_version."
    assert FROZEN_GOLD_ANNOTATION_VERSION, "Missing frozen Gold annotation version."
    assert GOLD_ANNOTATION_VERSION == [FROZEN_GOLD_ANNOTATION_VERSION]


if FORMAL_MODE:
    assert_formal_ready()
else:
    print(
        "Gold accepted for M6 scoring after verified evidence rebinding; "
        "the derived 273/68 split remains diagnostic-only."
    )
    try:
        assert_formal_ready()
    except AssertionError as exc:
        print("Formal gate correctly rejects these inputs:", exc)
    else:
        raise AssertionError("Formal gate unexpectedly accepted development inputs")

# Deterministic group-stratified 80/20 split. Questions sharing any rebound
# evidence chunk form one indivisible group so exact Golden Evidence cannot leak
# between Dev and Test.
TEST_QUOTAS = {
    "openstax_college_physics_2e": 25,
    "openstax_college_physics_ap_2e": 19,
    "ck12_peoples_physics_concepts": 17,
    "openstax_physics": 7,
}
SPLIT_SEED = "m6-v2-gold-v0.1.3-stratified-80-20"
STRATIFY_FIELDS = ("question_type", "question_difficulty", "source_split")


def question_textbook_id(question: dict) -> str:
    textbook_ids = {
        by_chunk[chunk_id]["textbook_id"]
        for path in gold_paths[question["question_id"]]
        for chunk_id in path
    }
    assert len(textbook_ids) == 1, (question["question_id"], textbook_ids)
    return next(iter(textbook_ids))


question_by_id = {g["question_id"]: g for g in gold}
parent = {question_id: question_id for question_id in question_by_id}


def find(question_id: str) -> str:
    while parent[question_id] != question_id:
        parent[question_id] = parent[parent[question_id]]
        question_id = parent[question_id]
    return question_id


def union(left: str, right: str) -> None:
    left_root, right_root = find(left), find(right)
    if left_root != right_root:
        parent[right_root] = left_root


questions_by_evidence_chunk = collections.defaultdict(list)
for question in gold:
    evidence_chunks = set().union(*gold_paths[question["question_id"]])
    for chunk_id in evidence_chunks:
        questions_by_evidence_chunk[chunk_id].append(question["question_id"])
for question_ids in questions_by_evidence_chunk.values():
    for other_id in question_ids[1:]:
        union(question_ids[0], other_id)

component_rows = collections.defaultdict(list)
for question in gold:
    component_rows[find(question["question_id"])].append(question)
components = list(component_rows.values())
assert sum(map(len, components)) == len(gold)
assert all(len({question_textbook_id(q) for q in group}) == 1 for group in components)


def component_hash(group: list[dict]) -> str:
    question_ids = "|".join(sorted(q["question_id"] for q in group))
    return hashlib.sha256(f"{SPLIT_SEED}|{question_ids}".encode()).hexdigest()


def select_test_components(textbook_id: str, quota: int) -> list[list[dict]]:
    candidates = [group for group in components if question_textbook_id(group[0]) == textbook_id]
    population = [question for group in candidates for question in group]
    population_counts = {
        field: collections.Counter(question[field] for question in population)
        for field in STRATIFY_FIELDS
    }
    selected, selected_rows = [], []
    remaining = list(candidates)
    while len(selected_rows) < quota:
        room = quota - len(selected_rows)
        feasible = [group for group in remaining if len(group) <= room]
        assert feasible, (textbook_id, room)

        def imbalance(group: list[dict]) -> tuple[float, str]:
            proposed = selected_rows + group
            penalty = 0.0
            for field in STRATIFY_FIELDS:
                observed = collections.Counter(question[field] for question in proposed)
                for value, population_count in population_counts[field].items():
                    target = len(proposed) * population_count / len(population)
                    penalty += (observed[value] - target) ** 2 / max(target, 1.0)
            return penalty, component_hash(group)

        chosen = min(feasible, key=imbalance)
        selected.append(chosen)
        selected_rows.extend(chosen)
        remaining.remove(chosen)
    assert len(selected_rows) == quota
    return selected


test_components = []
for textbook_id, quota in TEST_QUOTAS.items():
    test_components.extend(select_test_components(textbook_id, quota))
test_question_ids = {q["question_id"] for group in test_components for q in group}
assert len(test_question_ids) == sum(TEST_QUOTAS.values()) == 68
TEST_QUESTION_IDS_SHA256 = sha256("\n".join(sorted(test_question_ids)).encode())

dev_gold = [
    {**g, "source_gold_split": g["split"], "evaluation_split": "dev"}
    for g in gold
    if g["question_id"] not in test_question_ids
]
heldout_gold = [
    {**g, "source_gold_split": g["split"], "evaluation_split": "diagnostic_holdout"}
    for g in gold
    if g["question_id"] in test_question_ids
]
assert len(dev_gold) == 273 and len(heldout_gold) == 68
assert not ({g["question_id"] for g in dev_gold} & {g["question_id"] for g in heldout_gold})
assert collections.Counter(question_textbook_id(g) for g in heldout_gold) == collections.Counter(
    TEST_QUOTAS
)
assert {g["question_type"] for g in heldout_gold} == {
    "definition",
    "identification",
    "causal",
    "application",
    "comparison",
    "calculation",
}
dev_evidence = {
    chunk_id for q in dev_gold for path in gold_paths[q["question_id"]] for chunk_id in path
}
test_evidence = {
    chunk_id for q in heldout_gold for path in gold_paths[q["question_id"]] for chunk_id in path
}
assert not (dev_evidence & test_evidence), "Golden Evidence leakage across the derived split."
print("Diagnostic development:", len(dev_gold), "exposed holdout:", len(heldout_gold))
print(
    "Holdout textbooks:", dict(collections.Counter(question_textbook_id(g) for g in heldout_gold))
)
print(
    "Holdout question types:", dict(collections.Counter(g["question_type"] for g in heldout_gold))
)
print(
    "Holdout difficulties:",
    dict(collections.Counter(g["question_difficulty"] for g in heldout_gold)),
)
print("Shared evidence chunks across splits:", len(dev_evidence & test_evidence))

Gold accepted for M6 scoring after verified evidence rebinding; the derived 273/68 split remains diagnostic-only.
Formal gate correctly rejects these inputs: M4 corpus is development-only.
Diagnostic development: 273 exposed holdout: 68
Holdout textbooks: {'openstax_college_physics_2e': 25, 'openstax_college_physics_ap_2e': 19, 'openstax_physics': 7, 'ck12_peoples_physics_concepts': 17}
Holdout question types: {'identification': 21, 'causal': 17, 'definition': 22, 'comparison': 2, 'application': 5, 'calculation': 1}
Holdout difficulties: {'easy': 41, 'medium': 24, 'hard': 3}
Shared evidence chunks across splits: 0


## 5. Validate M5 indexes and prepare project-retriever views


In [5]:
M5_ARTIFACTS = {}
M5_INDEX_MEMBERS = {}
with zipfile.ZipFile(M5_ZIP) as archive:
    members = {name.replace("\\", "/"): name for name in archive.namelist()}
    assert len(members) == len(archive.namelist())
    for key, spec in MODEL_SPECS.items():
        if "m5_path" not in spec:
            continue
        prefix = f"{spec['m5_path']}/"
        artifact = json.loads(archive.read(members[prefix + "artifact.json"]))
        ids_bytes = archive.read(members[prefix + "index/chunk_ids.json"])
        index_member = members[prefix + "index/index.faiss"]
        index_bytes = archive.read(index_member)
        assert archive.read(members[prefix + "records.jsonl"]) == corpus_bytes
        assert json.loads(archive.read(members[prefix + "manifest.json"])) == corpus_manifest
        for name in ("corpus_version", "corpus_hash", "manifest_hash", "chunk_config_hash"):
            assert artifact[name] == corpus_manifest[name], (key, name)
        assert artifact["chunk_ids"] == ordered_chunk_ids
        assert json.loads(ids_bytes) == ordered_chunk_ids
        assert sha256(ids_bytes) == artifact["metadata"]["chunk_ids_sha256"]
        assert sha256(index_bytes) == artifact["metadata"]["index_sha256"]
        assert artifact["embedding_model"] == spec["model"]
        assert artifact["embedding_revision"] == spec["revision"]
        assert artifact["embedding_dimension"] == spec["dimension"]
        assert artifact["similarity_metric"] == "inner_product"
        assert artifact["normalise_embeddings"] is True
        M5_ARTIFACTS[key] = artifact
        M5_INDEX_MEMBERS[key] = index_member

display(
    pd.DataFrame(
        [
            {
                "model_key": key,
                "embedding_model": artifact["embedding_model"],
                "revision": artifact["embedding_revision"],
                "dimension": artifact["embedding_dimension"],
            }
            for key, artifact in M5_ARTIFACTS.items()
        ]
    )
)

,model_key,embedding_model,revision,dimension
0,gte_modernbert,Alibaba-NLP/gte-modernbert-base,e7f32e3c00f91d699e8c43b53106206bcc72bb22,768
1,bge_m3,BAAI/bge-m3,5617a9f61b028005a4858fdac845db406aefb181,1024
2,qwen3,Qwen/Qwen3-Embedding-0.6B,97b0c614be4d77ee51c0cef4e5f07c00f9eb65b3,1024


In [6]:
def local_v2_artifact(key: str, index_path: Path) -> dict:
    spec = MODEL_SPECS[key]
    return {
        "schema_version": "2.0",
        "artifact_id": f"m6-four-book-{key}-{corpus_manifest['corpus_hash'][-12:]}",
        "index_type": "faiss.IndexFlatIP",
        "corpus_hash": corpus_manifest["corpus_hash"],
        "chunk_config_hash": corpus_manifest["chunk_config_hash"],
        "chunk_count": len(chunks),
        "chunk_ids": ordered_chunk_ids,
        "embedding_model": spec["model"],
        "embedding_revision": spec["revision"],
        "embedding_dimension": spec["dimension"],
        "similarity_metric": "inner_product",
        "normalise_embeddings": True,
        "index_version": "m6-four-book-faiss-flat-ip-2",
        "metadata": {"index_sha256": sha256(index_path.read_bytes())},
    }


def ensure_raw_index(key: str) -> tuple[dict, Path]:
    spec = MODEL_SPECS[key]
    if key in M5_ARTIFACTS:
        raw_dir = WORK_DIR / "raw_indexes_shared" / key
        raw_dir.mkdir(parents=True, exist_ok=True)
        index_path = raw_dir / "index.faiss"
        expected_hash = M5_ARTIFACTS[key]["metadata"]["index_sha256"]
        if not index_path.is_file() or sha256(index_path.read_bytes()) != expected_hash:
            with zipfile.ZipFile(M5_ZIP) as archive:
                index_path.write_bytes(archive.read(M5_INDEX_MEMBERS[key]))
        assert sha256(index_path.read_bytes()) == expected_hash
        return M5_ARTIFACTS[key], index_path

    location = INDEX_DIR / key
    artifact_path = location / "artifact.json"
    index_path = location / "index.faiss"
    expected = {
        "corpus_hash": corpus_manifest["corpus_hash"],
        "model": spec["model"],
        "embedding_revision": spec["revision"],
        "query_prefix": spec["query_instruction"],
        "document_prefix": spec["document_prefix"],
        "metric": "inner_product",
        "normalise_embeddings": True,
        "chunk_ids_sha256": sha256(json.dumps(ordered_chunk_ids, separators=(",", ":")).encode()),
        "dimension": spec["dimension"],
        "chunk_count": len(chunks),
    }
    reusable = False
    if artifact_path.is_file() and index_path.is_file():
        saved = json.loads(artifact_path.read_text(encoding="utf-8"))
        reusable = all(saved.get(name) == value for name, value in expected.items())
        reusable = reusable and saved.get("index_sha256") == sha256(index_path.read_bytes())
    if not reusable:
        import faiss
        from sentence_transformers import SentenceTransformer

        model = SentenceTransformer(
            spec["model"],
            revision=spec["revision"],
            device=DEVICE,
            trust_remote_code=spec["trust_remote_code"],
            cache_folder=str(MODEL_CACHE),
        )
        texts = [spec["document_prefix"] + (row.get("embed_text") or row["text"]) for row in chunks]
        vectors = model.encode(
            texts,
            batch_size=spec["batch"],
            normalize_embeddings=True,
            convert_to_numpy=True,
            show_progress_bar=True,
        ).astype("float32")
        assert vectors.shape == (len(chunks), spec["dimension"])
        index = faiss.IndexFlatIP(spec["dimension"])
        index.add(vectors)
        location.mkdir(parents=True, exist_ok=True)
        faiss.write_index(index, str(index_path))
        saved = {
            **expected,
            "index_sha256": sha256(index_path.read_bytes()),
            "built_at_utc": datetime.now(UTC).isoformat(),
        }
        artifact_path.write_text(json.dumps(saved, indent=2) + "\n", encoding="utf-8")
        del model, vectors, index
        gc.collect()
    assert sha256(index_path.read_bytes()) == saved["index_sha256"]
    return local_v2_artifact(key, index_path), index_path


def prepare_retrieval_view(key: str) -> tuple[Path, dict]:
    artifact, index_path = ensure_raw_index(key)
    view_dir = WORK_DIR / "project_views_shared" / key
    spec = MODEL_SPECS[key]
    materialise_v2_retrieval_view(
        v2_artifact=artifact,
        records=chunks,
        index_source=index_path,
        output_dir=view_dir,
        query_instruction=spec["query_instruction"],
        query_prompt_name=spec["query_prompt_name"],
    )
    return view_dir, artifact

## 6. Truncation diagnostics


In [7]:
def truncation_diagnostic(key: str) -> dict:
    from sentence_transformers import SentenceTransformer

    spec = MODEL_SPECS[key]
    model = SentenceTransformer(
        spec["model"],
        revision=spec["revision"],
        device=DEVICE,
        trust_remote_code=spec["trust_remote_code"],
    )
    max_length = int(model.max_seq_length)
    truncated = 0
    maximum_observed = 0
    batch_size = 128
    texts = [spec["document_prefix"] + (row.get("embed_text") or row["text"]) for row in chunks]
    for start in range(0, len(texts), batch_size):
        encoded = model.tokenizer(
            texts[start : start + batch_size],
            add_special_tokens=True,
            truncation=False,
        )["input_ids"]
        lengths = [len(token_ids) for token_ids in encoded]
        maximum_observed = max(maximum_observed, max(lengths, default=0))
        truncated += sum(length > max_length for length in lengths)
    row = {
        "model": key,
        "embedding_model": spec["model"],
        "embedding_revision": spec["revision"],
        "max_input_tokens": max_length,
        "maximum_observed_tokens": maximum_observed,
        "chunk_count": len(texts),
        "truncated_chunk_count": truncated,
        "truncated_chunk_rate": truncated / len(texts),
    }
    del model
    gc.collect()
    return row


if RUN_EXPENSIVE:
    truncation_summary = pd.DataFrame([truncation_diagnostic(key) for key in MODEL_SPECS])
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    write_csv_lf(truncation_summary, OUTPUT_DIR / "truncation_diagnostics.csv")
    display(truncation_summary)
else:
    print("Truncation diagnostics run with CS30_M6_RUN_MODELS=1.")

Truncation diagnostics run with CS30_M6_RUN_MODELS=1.


## 7. Shared project retriever


In [8]:
def build_project_retriever(
    key: str,
    mode: RetrievalMode,
    *,
    dense_weight: float = 0.10,
    bm25_weight: float = 0.90,
):
    view_dir, artifact = prepare_retrieval_view(key)
    spec = MODEL_SPECS[key]
    config = AppConfig(
        fixture_mode=False,
        retrieval=RetrievalConfig(
            mode=mode,
            index_dir=str(view_dir),
            rrf_k=RRF_K,
            rrf_input_top_k=RRF_INPUT_TOP_K,
            rrf_dense_weight=dense_weight,
            rrf_bm25_weight=bm25_weight,
            bm25_stopwords=True,
            expected_embedding_model=spec["model"],
            expected_embedding_revision=spec["revision"],
            embedding_trust_remote_code=spec["trust_remote_code"],
            query_prompt_name=spec["query_prompt_name"],
        ),
    )
    return build_real_retrieval_deps(config).retriever, artifact


def hit_pairs(result) -> list[tuple[str, float]]:
    return [(hit.chunk_id, float(hit.score)) for hit in result.hits]


def candidate_rows(question: dict, model: str, mode: str, result) -> list[dict]:
    rows = []
    for hit in result.hits:
        chunk = by_chunk[hit.chunk_id]
        rows.append(
            {
                "question_id": question["question_id"],
                "model": model,
                "mode": mode,
                "rank": hit.rank,
                "chunk_id": hit.chunk_id,
                "score": float(hit.score),
                "query": question["question"],
                "textbook_id": chunk["textbook_id"],
                "provider": chunk["provider"],
                "document_id": chunk["document_id"],
                "chapter_id": chunk["chapter_id"],
                "source_name": chunk["source_name"],
                "page_or_location": chunk["page_or_location"],
                "source_locator": chunk["source_locator"],
                "text": chunk["text"],
            }
        )
    return rows

## 8. Evidence metrics


In [9]:
def score_one(
    question: dict,
    hits: list[tuple[str, float]],
    model_key: str,
    mode: str,
    bm25_weight=None,
    dense_weight=None,
) -> dict:
    ids = [chunk_id for chunk_id, _ in hits]
    assert len(ids) == len(set(ids)), (question["question_id"], model_key, mode, "duplicate hits")
    qid = question["question_id"]
    answerable = question.get("answerable", True) is True
    paths = gold_paths[qid] if answerable else []
    relevant = set().union(*paths) if paths else set()
    rank = {chunk_id: i for i, chunk_id in enumerate(ids, 1)}
    first_rank = min((rank[cid] for cid in relevant if cid in rank), default=None)
    complete_rank = min(
        (max((rank.get(cid, math.inf) for cid in path), default=math.inf) for path in paths),
        default=math.inf,
    )
    complete_rank = None if math.isinf(complete_rank) else int(complete_rank)
    source_ids = {by_chunk[cid]["textbook_id"] for cid in relevant}
    providers = {by_chunk[cid]["provider"] for cid in relevant}
    row = {
        "question_id": qid,
        "source_split": question["source_split"],
        "evaluation_split": question.get("evaluation_split", question["split"]),
        "source_gold_split": question.get("source_gold_split", question["split"]),
        "annotation_status": question["annotation_status"],
        "model": model_key,
        "mode": mode,
        "bm25_weight": bm25_weight,
        "dense_weight": dense_weight,
        "answerable": answerable,
        "scope": (
            "unanswerable"
            if not answerable
            else "cross_textbook"
            if len(source_ids) > 1
            else "single_source"
        ),
        "provider_scope": (
            "unanswerable"
            if not answerable
            else "cross_provider"
            if len(providers) > 1
            else "single_provider"
        ),
        "gold_textbook": "+".join(sorted(source_ids)),
        "gold_provider": "+".join(sorted(providers)),
        "first_relevant_rank": first_rank,
        "complete_evidence_rank": complete_rank,
        "mrr": 0.0 if first_rank is None else 1 / first_rank,
        "complete_evidence_mrr": 0.0 if complete_rank is None else 1 / complete_rank,
        "returned_count": len(hits),
        "empty_retrieval": int(not hits),
        "unanswerable_no_hit": int(not hits) if not answerable else None,
        "unanswerable_false_positive": int(bool(hits)) if not answerable else None,
        "ranked_chunk_ids": ids,
    }
    for k in TOP_K:
        top = set(ids[:k])
        row[f"hit_at_{k}"] = int(bool(top & relevant))
        row[f"complete_hit_at_{k}"] = int(any(path <= top for path in paths))
        row[f"evidence_recall_at_{k}"] = (
            max((len(path & top) / len(path) for path in paths), default=0.0)
            if answerable
            else None
        )
        row[f"core_precision_at_{k}"] = (
            len(top & relevant) / len(top) if top and answerable else None
        )
        row[f"no_relevant_hit_at_{k}"] = int(not (top & relevant)) if answerable else None
    return row


def summarize(rows: list[dict]) -> pd.DataFrame:
    frame = pd.DataFrame(rows)
    # The integrated Gold has no partial-evidence spans. Thus noise is exactly
    # 1 - core precision for nonempty retrieval, and partial count is zero.
    assert all(not q.get("partial_evidence") for q in gold)
    frame["run_name"] = frame["model"] + "_" + frame["mode"]
    frame["first_hit_mrr"] = frame["mrr"]
    for k in TOP_K:
        frame[f"complete_evidence_hit_at_{k}"] = frame[f"complete_hit_at_{k}"]
        frame[f"recall_at_{k}"] = frame[f"evidence_recall_at_{k}"]
        frame[f"returned_count_at_{k}"] = frame["returned_count"].clip(upper=k)
        frame[f"empty_retrieval_rate_at_{k}"] = frame[f"returned_count_at_{k}"].eq(0).astype(float)
        frame[f"no_hit_rate_at_{k}"] = frame[f"empty_retrieval_rate_at_{k}"]
        frame[f"no_relevant_hit_rate_at_{k}"] = frame[f"no_relevant_hit_at_{k}"].where(
            frame[f"returned_count_at_{k}"].gt(0)
        )
        frame[f"context_noise_rate_at_{k}"] = 1 - frame[f"core_precision_at_{k}"]
        frame[f"partial_evidence_count_at_{k}"] = 0
    metrics = (
        ["mrr", "complete_evidence_mrr"]
        + [
            f"{metric}_at_{k}"
            for metric in ("hit", "complete_evidence_hit", "evidence_recall")
            for k in TOP_K
        ]
        + ["first_hit_mrr"]
        + [
            f"{metric}_at_{k}"
            for metric in (
                "complete_hit",
                "recall",
                "core_precision",
                "context_noise_rate",
                "no_relevant_hit_rate",
                "partial_evidence_count",
                "returned_count",
                "empty_retrieval_rate",
                "no_hit_rate",
            )
            for k in TOP_K
        ]
        + [
            "returned_count",
            "empty_retrieval",
            "unanswerable_no_hit",
            "unanswerable_false_positive",
        ]
        + [f"no_relevant_hit_at_{k}" for k in TOP_K]
    )
    groups = ["run_name", "model", "mode", "dense_weight", "bm25_weight"]
    grouped = frame.groupby(groups, dropna=False)
    counts = grouped.size().rename("question_count")
    nullable = [
        f"{metric}_at_{k}"
        for metric in (
            "core_precision",
            "context_noise_rate",
            "no_relevant_hit_rate",
            "partial_evidence_count",
        )
        for k in TOP_K
    ]
    defined = (
        grouped[nullable]
        .count()
        .rename(columns={column: f"{column}_defined_count" for column in nullable})
    )
    return counts.to_frame().join(grouped[metrics].mean()).join(defined).reset_index()


def grouped_summary(rows: list[dict]) -> pd.DataFrame:
    frame = pd.DataFrame(rows)
    groups = [
        "model",
        "mode",
        "evaluation_split",
        "scope",
        "provider_scope",
        "gold_textbook",
        "gold_provider",
    ]
    return (
        frame.groupby(groups, dropna=False)
        .agg(
            question_count=("question_id", "nunique"),
            hit_at_5=("hit_at_5", "mean"),
            complete_hit_at_5=("complete_hit_at_5", "mean"),
            evidence_recall_at_5=("evidence_recall_at_5", "mean"),
            mrr=("mrr", "mean"),
        )
        .reset_index()
    )


# Tiny synthetic tests only verify metric logic; they are never merged with real output.
_q = {
    "question_id": dev_gold[0]["question_id"],
    "source_split": "train",
    "split": "pending",
    "annotation_status": dev_gold[0]["annotation_status"],
}
_a = next(iter(gold_paths[_q["question_id"]][0]))
assert score_one(_q, [(_a, 1.0)], "fixture", "fixture")["hit_at_1"] == 1
assert score_one(_q, [], "fixture", "fixture")["empty_retrieval"] == 1
assert score_one(_q, [], "fixture", "fixture")["no_relevant_hit_at_5"] == 1
_u = {**_q, "answerable": False}
assert score_one(_u, [], "fixture", "fixture")["unanswerable_no_hit"] == 1
assert score_one(_u, [(_a, 1.0)], "fixture", "fixture")["unanswerable_false_positive"] == 1
_joint = next(q for q in gold if any(len(path) > 1 for path in gold_paths[q["question_id"]]))
_joint_ids = list(next(path for path in gold_paths[_joint["question_id"]] if len(path) > 1))
assert score_one(_joint, [(_joint_ids[0], 1.0)], "fixture", "fixture")["hit_at_1"] == 1
assert score_one(_joint, [(_joint_ids[0], 1.0)], "fixture", "fixture")["complete_hit_at_1"] == 0
assert (
    score_one(_joint, [(cid, 1.0) for cid in _joint_ids], "fixture", "fixture")["complete_hit_at_3"]
    == 1
)
print("Metric smoke tests passed.")

Metric smoke tests passed.


## 9. Development sweep: 64 configurations


In [10]:
def write_checkpoint(path: Path, rows: list[dict]) -> None:
    write_utf8_lf(
        path,
        "".join(json.dumps(row, ensure_ascii=False, allow_nan=False) + "\n" for row in rows),
    )


def read_checkpoint(path: Path) -> list[dict]:
    return read_jsonl_bytes(path.read_bytes())


def evaluate_dev() -> tuple[list[dict], list[dict], dict]:
    rows: list[dict] = []
    candidates: list[dict] = []
    provenance: dict[str, dict] = {}

    baseline_metrics = OUTPUT_DIR / "dev_bm25_metrics.jsonl"
    baseline_candidates = OUTPUT_DIR / "dev_bm25_candidates.jsonl"
    if baseline_metrics.is_file() and baseline_candidates.is_file():
        recovered_rows = read_checkpoint(baseline_metrics)
        recovered_candidates = read_checkpoint(baseline_candidates)
        assert len(recovered_rows) == len(dev_gold)
        rows.extend(recovered_rows)
        candidates.extend(recovered_candidates)
        print("Recovered project-retriever BM25 checkpoint", flush=True)
    else:
        baseline, _ = build_project_retriever("bge_m3", RetrievalMode.BM25)
        baseline_rows = []
        baseline_hits = []
        for question in dev_gold:
            result = baseline.retrieve(question["question"], FINAL_TOP_K)
            baseline_rows.append(score_one(question, hit_pairs(result), "bm25", "bm25"))
            baseline_hits.extend(candidate_rows(question, "bm25", "bm25", result))
        write_checkpoint(baseline_metrics, baseline_rows)
        write_checkpoint(baseline_candidates, baseline_hits)
        rows.extend(baseline_rows)
        candidates.extend(baseline_hits)
        del baseline

    for key in MODEL_SPECS:
        metrics_path = OUTPUT_DIR / f"dev_{key}_metrics.jsonl"
        candidates_path = OUTPUT_DIR / f"dev_{key}_candidates.jsonl"
        if metrics_path.is_file() and candidates_path.is_file():
            model_rows = read_checkpoint(metrics_path)
            model_candidates = read_checkpoint(candidates_path)
            assert len(model_rows) == len(dev_gold) * (1 + len(HYBRID_WEIGHTS))
            _, artifact = prepare_retrieval_view(key)
            provenance[key] = artifact
            rows.extend(model_rows)
            candidates.extend(model_candidates)
            print("Recovered project-retriever checkpoint:", key, flush=True)
            continue

        print("Dev shared retriever:", key, flush=True)
        hybrid, artifact = build_project_retriever(key, RetrievalMode.HYBRID)
        provenance[key] = artifact
        model_rows = []
        model_candidates = []
        for question in dev_gold:
            dense_result = hybrid.dense.retrieve(question["question"], FINAL_TOP_K)
            model_rows.append(score_one(question, hit_pairs(dense_result), key, "dense"))
            model_candidates.extend(candidate_rows(question, key, "dense", dense_result))
            for bm25_weight, dense_weight in HYBRID_WEIGHTS:
                hybrid.bm25_weight = bm25_weight
                hybrid.dense_weight = dense_weight
                mode = f"hybrid_d{dense_weight:.2f}_b{bm25_weight:.2f}"
                result = hybrid.retrieve(question["question"], FINAL_TOP_K)
                model_rows.append(
                    score_one(
                        question,
                        hit_pairs(result),
                        key,
                        mode,
                        bm25_weight,
                        dense_weight,
                    )
                )
                model_candidates.extend(candidate_rows(question, key, mode, result))
        write_checkpoint(metrics_path, model_rows)
        write_checkpoint(candidates_path, model_candidates)
        rows.extend(model_rows)
        candidates.extend(model_candidates)
        del hybrid
        gc.collect()
        try:
            import torch

            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        except ImportError:
            pass
    return rows, candidates, provenance


def sort_summary(frame: pd.DataFrame) -> pd.DataFrame:
    return frame.sort_values(
        ["complete_hit_at_5", "mrr", "run_name"],
        ascending=[False, False, True],
    ).reset_index(drop=True)


if RUN_EXPENSIVE:
    from IPython.display import display

    run_git_state = capture_git_state(PROJECT_ROOT)
    assert not run_git_state.dirty, "Run from a clean committed worktree."
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    INDEX_DIR.mkdir(parents=True, exist_ok=True)
    MODEL_CACHE.mkdir(parents=True, exist_ok=True)
    run_identity_path = OUTPUT_DIR / "run_identity.json"
    stable_run_identity = {
        "corpus_hash": corpus_manifest["corpus_hash"],
        "gold_sha256": sha256(gold_bytes),
        "project_retriever": "cs30.pipeline.build_real_retrieval_deps",
        "rrf_input_top_k": RRF_INPUT_TOP_K,
    }
    if run_identity_path.is_file():
        cached_run_identity = json.loads(run_identity_path.read_text(encoding="utf-8"))
        checkpoint_generation_commit = cached_run_identity.pop(
            "checkpoint_generation_commit", cached_run_identity.pop("commit", None)
        )
        assert checkpoint_generation_commit
        assert cached_run_identity == stable_run_identity
    else:
        checkpoint_generation_commit = run_git_state.commit
    run_identity = {
        "checkpoint_generation_commit": checkpoint_generation_commit,
        **stable_run_identity,
    }
    run_identity_path.write_text(json.dumps(run_identity, indent=2) + "\n", encoding="utf-8")
    dev_rows, dev_candidates, index_provenance = evaluate_dev()
    assert len(dev_rows) == len(dev_gold) * 64
    dev_summary = sort_summary(summarize(dev_rows))
    display(dev_summary)
else:
    dev_rows = dev_candidates = []
    index_provenance = {}
    print("Preflight only: shared-retriever model execution skipped.")

Preflight only: shared-retriever model execution skipped.


## 10. Development results by model


In [11]:
from IPython.display import display


def result_path(name: str) -> Path:
    local = OUTPUT_DIR / name
    published = PUBLISHED_RESULT_DIR / name
    if local.is_file():
        return local
    if published.is_file():
        return published
    return local


def model_table(key: str) -> pd.DataFrame:
    frame = (
        dev_summary if "dev_summary" in globals() else pd.read_csv(result_path("dev_summary.csv"))
    )
    table = frame.loc[frame["model"].eq(key)].copy()
    if table.empty:
        raise ValueError(f"No measured rows for {key}")
    return table.sort_values(["complete_hit_at_5", "mrr"], ascending=[False, False])


if "dev_summary" in globals() or result_path("dev_summary.csv").is_file():
    for model_key in ["bm25", *MODEL_SPECS]:
        print("Dev results:", model_key)
        display(model_table(model_key))
else:
    print("Dev results pending. Run the full notebook first.")

Dev results: bm25


,run_name,model,mode,dense_weight,bm25_weight,question_count,mrr,complete_evidence_mrr,hit_at_1,hit_at_3,...,context_noise_rate_at_5_defined_count,context_noise_rate_at_10_defined_count,no_relevant_hit_rate_at_1_defined_count,no_relevant_hit_rate_at_3_defined_count,no_relevant_hit_rate_at_5_defined_count,no_relevant_hit_rate_at_10_defined_count,partial_evidence_count_at_1_defined_count,partial_evidence_count_at_3_defined_count,partial_evidence_count_at_5_defined_count,partial_evidence_count_at_10_defined_count
6,bm25_bm25,bm25,bm25,NaN,NaN,273,0.575042,0.572905,0.399267,0.747253,...,273,273,273,273,273,273,273,273,273,273


Dev results: minilm


,run_name,model,mode,dense_weight,bm25_weight,question_count,mrr,complete_evidence_mrr,hit_at_1,hit_at_3,...,context_noise_rate_at_5_defined_count,context_noise_rate_at_10_defined_count,no_relevant_hit_rate_at_1_defined_count,no_relevant_hit_rate_at_3_defined_count,no_relevant_hit_rate_at_5_defined_count,no_relevant_hit_rate_at_10_defined_count,partial_evidence_count_at_1_defined_count,partial_evidence_count_at_3_defined_count,partial_evidence_count_at_5_defined_count,partial_evidence_count_at_10_defined_count
15,minilm_hybrid_d0.05_b0.95,minilm,hybrid_d0.05_b0.95,0.05,0.95,273,0.550158,0.546495,0.362637,0.721612,...,273,273,273,273,273,273,273,273,273,273
23,minilm_hybrid_d0.10_b0.90,minilm,hybrid_d0.10_b0.90,0.10,0.90,273,0.524545,0.521493,0.344322,0.666667,...,273,273,273,273,273,273,273,273,273,273
40,minilm_hybrid_d0.20_b0.80,minilm,hybrid_d0.20_b0.80,0.20,0.80,273,0.495828,0.492776,0.336996,0.597070,...,273,273,273,273,273,273,273,273,273,273
44,minilm_hybrid_d0.25_b0.75,minilm,hybrid_d0.25_b0.75,0.25,0.75,273,0.488537,0.485485,0.329670,0.586081,...,273,273,273,273,273,273,273,273,273,273
48,minilm_hybrid_d0.50_b0.50,minilm,hybrid_d0.50_b0.50,0.50,0.50,273,0.439687,0.435291,0.285714,0.516484,...,273,273,273,273,273,273,273,273,273,273
57,minilm_hybrid_d0.75_b0.25,minilm,hybrid_d0.75_b0.25,0.75,0.25,273,0.341168,0.337047,0.186813,0.446886,...,273,273,273,273,273,273,273,273,273,273
62,minilm_dense,minilm,dense,NaN,NaN,273,0.281771,0.277741,0.139194,0.391941,...,273,273,273,273,273,273,273,273,273,273


Dev results: mpnet


...

,run_name,model,mode,dense_weight,bm25_weight,question_count,mrr,complete_evidence_mrr,hit_at_1,hit_at_3,...,context_noise_rate_at_5_defined_count,context_noise_rate_at_10_defined_count,no_relevant_hit_rate_at_1_defined_count,no_relevant_hit_rate_at_3_defined_count,no_relevant_hit_rate_at_5_defined_count,no_relevant_hit_rate_at_10_defined_count,partial_evidence_count_at_1_defined_count,partial_evidence_count_at_3_defined_count,partial_evidence_count_at_5_defined_count,partial_evidence_count_at_10_defined_count
4,nomic_hybrid_d0.05_b0.95,nomic,hybrid_d0.05_b0.95,0.05,0.95,273,0.555598,0.553156,0.377289,0.703297,...,273,273,273,273,273,273,273,273,273,273
19,nomic_hybrid_d0.10_b0.90,nomic,hybrid_d0.10_b0.90,0.10,0.90,273,0.528244,0.525802,0.351648,0.655678,...,273,273,273,273,273,273,273,273,273,273
31,nomic_hybrid_d0.20_b0.80,nomic,hybrid_d0.20_b0.80,0.20,0.80,273,0.514784,0.512953,0.351648,0.622711,...,273,273,273,273,273,273,273,273,273,273
35,nomic_hybrid_d0.25_b0.75,nomic,hybrid_d0.25_b0.75,0.25,0.75,273,0.515629,0.513797,0.355311,0.622711,...,273,273,273,273,273,273,273,273,273,273
42,nomic_hybrid_d0.50_b0.50,nomic,hybrid_d0.50_b0.50,0.50,0.50,273,0.478355,0.476523,0.315018,0.571429,...,273,273,273,273,273,273,273,273,273,273
50,nomic_hybrid_d0.75_b0.25,nomic,hybrid_d0.75_b0.25,0.75,0.25,273,0.388976,0.385679,0.216117,0.509158,...,273,273,273,273,273,273,273,273,273,273
58,nomic_dense,nomic,dense,NaN,NaN,273,0.303787,0.302356,0.131868,0.417582,...,273,273,273,273,273,273,273,273,273,273


Dev results: bge_m3


,run_name,model,mode,dense_weight,bm25_weight,question_count,mrr,complete_evidence_mrr,hit_at_1,hit_at_3,...,context_noise_rate_at_5_defined_count,context_noise_rate_at_10_defined_count,no_relevant_hit_rate_at_1_defined_count,no_relevant_hit_rate_at_3_defined_count,no_relevant_hit_rate_at_5_defined_count,no_relevant_hit_rate_at_10_defined_count,partial_evidence_count_at_1_defined_count,partial_evidence_count_at_3_defined_count,partial_evidence_count_at_5_defined_count,partial_evidence_count_at_10_defined_count
1,bge_m3_hybrid_d0.10_b0.90,bge_m3,hybrid_d0.10_b0.90,0.10,0.90,273,0.545921,0.543566,0.358974,0.703297,...,273,273,273,273,273,273,273,273,273,273
7,bge_m3_hybrid_d0.05_b0.95,bge_m3,hybrid_d0.05_b0.95,0.05,0.95,273,0.567683,0.565119,0.384615,0.750916,...,273,273,273,273,273,273,273,273,273,273
21,bge_m3_hybrid_d0.20_b0.80,bge_m3,hybrid_d0.20_b0.80,0.20,0.80,273,0.528073,0.526241,0.351648,0.666667,...,273,273,273,273,273,273,273,273,273,273
25,bge_m3_hybrid_d0.25_b0.75,bge_m3,hybrid_d0.25_b0.75,0.25,0.75,273,0.523107,0.521276,0.344322,0.659341,...,273,273,273,273,273,273,273,273,273,273
29,bge_m3_hybrid_d0.50_b0.50,bge_m3,hybrid_d0.50_b0.50,0.50,0.50,273,0.471013,0.467350,0.285714,0.608059,...,273,273,273,273,273,273,273,273,273,273
46,bge_m3_hybrid_d0.75_b0.25,bge_m3,hybrid_d0.75_b0.25,0.75,0.25,273,0.400677,0.397014,0.219780,0.527473,...,273,273,273,273,273,273,273,273,273,273
52,bge_m3_dense,bge_m3,dense,NaN,NaN,273,0.354790,0.351493,0.183150,0.468864,...,273,273,273,273,273,273,273,273,273,273


Dev results: qwen3


,run_name,model,mode,dense_weight,bm25_weight,question_count,mrr,complete_evidence_mrr,hit_at_1,hit_at_3,...,context_noise_rate_at_5_defined_count,context_noise_rate_at_10_defined_count,no_relevant_hit_rate_at_1_defined_count,no_relevant_hit_rate_at_3_defined_count,no_relevant_hit_rate_at_5_defined_count,no_relevant_hit_rate_at_10_defined_count,partial_evidence_count_at_1_defined_count,partial_evidence_count_at_3_defined_count,partial_evidence_count_at_5_defined_count,partial_evidence_count_at_10_defined_count
2,qwen3_hybrid_d0.05_b0.95,qwen3,hybrid_d0.05_b0.95,0.05,0.95,273,0.568835,0.566271,0.391941,0.736264,...,273,273,273,273,273,273,273,273,273,273
3,qwen3_hybrid_d0.10_b0.90,qwen3,hybrid_d0.10_b0.90,0.10,0.90,273,0.556519,0.554164,0.373626,0.710623,...,273,273,273,273,273,273,273,273,273,273
16,qwen3_hybrid_d0.25_b0.75,qwen3,hybrid_d0.25_b0.75,0.25,0.75,273,0.535463,0.533265,0.355311,0.692308,...,273,273,273,273,273,273,273,273,273,273
17,qwen3_hybrid_d0.20_b0.80,qwen3,hybrid_d0.20_b0.80,0.20,0.80,273,0.539799,0.537601,0.358974,0.703297,...,273,273,273,273,273,273,273,273,273,273
24,qwen3_hybrid_d0.50_b0.50,qwen3,hybrid_d0.50_b0.50,0.50,0.50,273,0.503928,0.501730,0.304029,0.663004,...,273,273,273,273,273,273,273,273,273,273
43,qwen3_hybrid_d0.75_b0.25,qwen3,hybrid_d0.75_b0.25,0.75,0.25,273,0.446007,0.443260,0.267399,0.578755,...,273,273,273,273,273,273,273,273,273,273
53,qwen3_dense,qwen3,dense,NaN,NaN,273,0.388899,0.386600,0.227106,0.498168,...,273,273,273,273,273,273,273,273,273,273


## 11. Dev-only baseline decision


In [12]:
def exact_mcnemar_p(baseline_only: int, candidate_only: int) -> float:
    discordant = baseline_only + candidate_only
    if discordant == 0:
        return 1.0
    tail = sum(
        math.comb(discordant, successes)
        for successes in range(min(baseline_only, candidate_only) + 1)
    ) / (2**discordant)
    return min(1.0, 2 * tail)


def compare_candidate_to_baseline(
    candidate: pd.Series,
    candidate_role: str,
    selection_basis: str,
    baseline_summary: pd.Series,
    baseline_by_question: dict[str, dict],
) -> dict:
    candidate_by_question = {
        row["question_id"]: row
        for row in dev_rows
        if row["model"] == candidate["model"] and row["mode"] == candidate["mode"]
    }
    assert baseline_by_question.keys() == candidate_by_question.keys()
    baseline_only = sum(
        baseline_by_question[question_id]["complete_hit_at_5"]
        > candidate_by_question[question_id]["complete_hit_at_5"]
        for question_id in baseline_by_question
    )
    candidate_only = sum(
        candidate_by_question[question_id]["complete_hit_at_5"]
        > baseline_by_question[question_id]["complete_hit_at_5"]
        for question_id in baseline_by_question
    )
    mcnemar_p = exact_mcnemar_p(baseline_only, candidate_only)
    passes_rule = bool(
        candidate["complete_hit_at_5"] > baseline_summary["complete_hit_at_5"]
        and candidate["mrr"] >= baseline_summary["mrr"]
        and mcnemar_p < 0.05
    )
    return {
        "comparison_role": candidate_role,
        "baseline_run_name": baseline_summary["run_name"],
        "candidate_run_name": candidate["run_name"],
        "candidate_selection_basis": selection_basis,
        "question_count": len(baseline_by_question),
        "baseline_complete_hits_at_5": int(
            sum(row["complete_hit_at_5"] for row in baseline_by_question.values())
        ),
        "candidate_complete_hits_at_5": int(
            sum(row["complete_hit_at_5"] for row in candidate_by_question.values())
        ),
        "baseline_mrr": float(baseline_summary["mrr"]),
        "candidate_mrr": float(candidate["mrr"]),
        "baseline_only_hits": baseline_only,
        "candidate_only_hits": candidate_only,
        "exact_mcnemar_p": mcnemar_p,
        "passes_baseline_replacement_rule": passes_rule,
        "selection_decision": "replace_bm25" if passes_rule else "retain_bm25",
        "selection_reason": (
            "candidate passed the preregistered Dev replacement rule"
            if passes_rule
            else "candidate did not pass the preregistered Dev replacement rule"
        ),
    }


if RUN_EXPENSIVE:
    metric_winner = dev_summary.loc[dev_summary["mode"].str.startswith("hybrid_")].iloc[0]
    system_rows = dev_summary.loc[
        dev_summary["model"].eq(SYSTEM_RETRIEVER_MODEL)
        & dev_summary["mode"].eq(SYSTEM_RETRIEVER_MODE)
    ]
    assert len(system_rows) == 1
    system_retriever = system_rows.iloc[0]
    baseline_by_question = {
        row["question_id"]: row
        for row in dev_rows
        if row["model"] == "bm25" and row["mode"] == "bm25"
    }
    baseline_summary = dev_summary.loc[dev_summary["run_name"] == "bm25_bm25"].iloc[0]
    baseline_comparison = pd.DataFrame(
        [
            compare_candidate_to_baseline(
                metric_winner,
                "dev_metric_winner",
                "highest complete_hit_at_5, then highest mrr, among Dev hybrids",
                baseline_summary,
                baseline_by_question,
            ),
            compare_candidate_to_baseline(
                system_retriever,
                "system_retriever_joint_robustness_rule",
                SYSTEM_SELECTION_BASIS,
                baseline_summary,
                baseline_by_question,
            ),
        ]
    )
    formal_baseline_selection = baseline_comparison.loc[
        baseline_comparison["comparison_role"].eq("dev_metric_winner")
    ].iloc[0]
    system_baseline_comparison = baseline_comparison.loc[
        baseline_comparison["comparison_role"].eq("system_retriever_joint_robustness_rule")
    ].iloc[0]
    assert formal_baseline_selection["candidate_run_name"] == metric_winner["run_name"]
    assert system_baseline_comparison["candidate_run_name"] == system_retriever["run_name"]
    write_csv_lf(baseline_comparison, OUTPUT_DIR / "dev_baseline_comparison.csv")
    display(baseline_comparison)

    counterpart_dir = PARA_OUTPUT_DIR if QUERY_SET == "original" else ORIGINAL_OUTPUT_DIR
    counterpart_summary_path = counterpart_dir / "dev_summary.csv"
    if counterpart_summary_path.is_file():
        original_summary = (
            dev_summary.copy()
            if QUERY_SET == "original"
            else pd.read_csv(ORIGINAL_OUTPUT_DIR / "dev_summary.csv")
        )
        para_summary = (
            dev_summary.copy()
            if QUERY_SET == "para_v1"
            else pd.read_csv(PARA_OUTPUT_DIR / "dev_summary.csv")
        )
        joint_selection = original_summary[
            [
                "run_name",
                "model",
                "mode",
                "dense_weight",
                "bm25_weight",
                "question_count",
                "complete_hit_at_5",
                "mrr",
            ]
        ].merge(
            para_summary[
                [
                    "run_name",
                    "question_count",
                    "complete_hit_at_5",
                    "mrr",
                ]
            ],
            on="run_name",
            suffixes=("_original", "_para_v1"),
            validate="one_to_one",
        )
        joint_selection["combined_question_count"] = (
            joint_selection["question_count_original"] + joint_selection["question_count_para_v1"]
        )
        joint_selection["combined_complete_hits_at_5"] = (
            (
                joint_selection["complete_hit_at_5_original"]
                * joint_selection["question_count_original"]
                + joint_selection["complete_hit_at_5_para_v1"]
                * joint_selection["question_count_para_v1"]
            )
            .round()
            .astype(int)
        )
        joint_selection["combined_complete_hit_at_5"] = (
            joint_selection["combined_complete_hits_at_5"]
            / joint_selection["combined_question_count"]
        )
        joint_selection["mean_mrr"] = (
            joint_selection["mrr_original"] * joint_selection["question_count_original"]
            + joint_selection["mrr_para_v1"] * joint_selection["question_count_para_v1"]
        ) / joint_selection["combined_question_count"]
        joint_selection = joint_selection.sort_values(
            ["combined_complete_hit_at_5", "mean_mrr", "run_name"],
            ascending=[False, False, True],
        ).reset_index(drop=True)
        joint_selection.insert(0, "rule1_rank", range(1, len(joint_selection) + 1))
        joint_selection["rule1_metric_winner"] = joint_selection["rule1_rank"].eq(1)
        joint_selection["selected_system_retriever"] = joint_selection["run_name"].eq(
            f"{SYSTEM_RETRIEVER_MODEL}_{SYSTEM_RETRIEVER_MODE}"
        )
        system_joint_row = joint_selection.loc[joint_selection["selected_system_retriever"]].iloc[0]
        complete_hit_gap = int(
            joint_selection.iloc[0]["combined_complete_hits_at_5"]
            - system_joint_row["combined_complete_hits_at_5"]
        )
        assert complete_hit_gap <= SYSTEM_MAX_COMPLETE_HIT_GAP
        joint_selection["system_selection_note"] = ""
        joint_selection.loc[
            joint_selection["selected_system_retriever"],
            "system_selection_note",
        ] = (
            "team-selected system retriever for M7 handoff; not the rule-1 metric winner"
            if int(system_joint_row["rule1_rank"]) != 1
            else "team-selected system retriever and rule-1 metric winner"
        )
        for destination in (ORIGINAL_OUTPUT_DIR, PARA_OUTPUT_DIR):
            destination.mkdir(parents=True, exist_ok=True)
            write_csv_lf(joint_selection, destination / "dev_rule1_selection.csv")

        def rows_for(query_set: str, model: str, mode: str) -> list[dict]:
            if query_set == QUERY_SET:
                return [row for row in dev_rows if row["model"] == model and row["mode"] == mode]
            source_dir = ORIGINAL_OUTPUT_DIR if query_set == "original" else PARA_OUTPUT_DIR
            filename = f"dev_{model}_metrics.jsonl"
            return [
                row
                for row in read_jsonl_bytes((source_dir / filename).read_bytes())
                if row["model"] == model and row["mode"] == mode
            ]

        paired_detail = []
        paired_summary = []
        for query_set in ("original", "para_v1"):
            baseline_rows = rows_for(query_set, "bm25", "bm25")
            candidate_rows_for_set = rows_for(
                query_set, SYSTEM_RETRIEVER_MODEL, SYSTEM_RETRIEVER_MODE
            )
            baseline = {row["question_id"]: row for row in baseline_rows}
            candidate = {row["question_id"]: row for row in candidate_rows_for_set}
            assert baseline.keys() == candidate.keys() and len(baseline) == 273
            question_ids = sorted(baseline)
            baseline_hit = np.array([baseline[q]["complete_hit_at_5"] for q in question_ids])
            candidate_hit = np.array([candidate[q]["complete_hit_at_5"] for q in question_ids])
            baseline_mrr = np.array([baseline[q]["mrr"] for q in question_ids], dtype=float)
            candidate_mrr = np.array([candidate[q]["mrr"] for q in question_ids], dtype=float)
            baseline_only = int(((baseline_hit == 1) & (candidate_hit == 0)).sum())
            candidate_only = int(((baseline_hit == 0) & (candidate_hit == 1)).sum())
            difference = candidate_mrr - baseline_mrr
            rng = np.random.default_rng(20261023 + (query_set == "para_v1"))
            samples = difference[
                rng.integers(0, len(difference), size=(20000, len(difference)))
            ].mean(axis=1)
            paired_summary.append(
                {
                    "query_set": query_set,
                    "question_count": len(question_ids),
                    "baseline_complete_hits_at_5": int(baseline_hit.sum()),
                    "candidate_complete_hits_at_5": int(candidate_hit.sum()),
                    "baseline_only_hits": baseline_only,
                    "candidate_only_hits": candidate_only,
                    "exact_mcnemar_p": exact_mcnemar_p(baseline_only, candidate_only),
                    "baseline_mrr": float(baseline_mrr.mean()),
                    "candidate_mrr": float(candidate_mrr.mean()),
                    "mrr_difference": float(difference.mean()),
                    "mrr_difference_bootstrap_95_ci_low": float(np.quantile(samples, 0.025)),
                    "mrr_difference_bootstrap_95_ci_high": float(np.quantile(samples, 0.975)),
                }
            )
            for model_name, mode_name, rows_by_id in (
                ("bm25", "bm25", baseline),
                (SYSTEM_RETRIEVER_MODEL, SYSTEM_RETRIEVER_MODE, candidate),
            ):
                for question_id in question_ids:
                    row = rows_by_id[question_id]
                    paired_detail.append(
                        {
                            "query_set": query_set,
                            "question_id": question_id,
                            "model": model_name,
                            "mode": mode_name,
                            "complete_hit_at_5": row["complete_hit_at_5"],
                            "hit_at_5": row["hit_at_5"],
                            "mrr": row["mrr"],
                            "complete_evidence_rank": row["complete_evidence_rank"],
                            "first_relevant_rank": row["first_relevant_rank"],
                        }
                    )
        paired_summary_frame = pd.DataFrame(paired_summary)
        paired_detail_frame = pd.DataFrame(paired_detail)
        for destination in (ORIGINAL_OUTPUT_DIR, PARA_OUTPUT_DIR):
            write_csv_lf(
                paired_summary_frame,
                destination / "dev_paired_bm25_vs_bge_m3_summary.csv",
            )
            write_csv_lf(
                paired_detail_frame,
                destination / "dev_paired_bm25_vs_bge_m3_questions.csv",
            )
        display(joint_selection.head(15))
        display(paired_summary_frame)
    else:
        print("Joint selection table pending the other query-set run.")
else:
    print("Dev baseline decision requires CS30_M6_RUN_MODELS=1.")

Dev baseline decision requires CS30_M6_RUN_MODELS=1.


## 12. Previously exposed diagnostic holdout

Only BM25 and the BGE-M3 Hybrid system retriever are evaluated here. The
system retriever was selected by the joint original/para-v1 Dev robustness
rule, not from these exposed questions. The
68 questions were exposed to all configurations in a superseded run, so these
numbers cannot be presented as a clean final Test result and cannot change the
Dev selection decision.


In [13]:
def evaluate_diagnostic_holdout() -> tuple[list[dict], list[dict]]:
    rows = []
    candidates = []
    hybrid, _ = build_project_retriever(
        diagnostic_configuration["model"],
        RetrievalMode.HYBRID,
        dense_weight=diagnostic_configuration["dense_weight"],
        bm25_weight=diagnostic_configuration["bm25_weight"],
    )
    for question in heldout_gold:
        baseline_result = hybrid.bm25.retrieve(question["question"], FINAL_TOP_K)
        rows.append(score_one(question, hit_pairs(baseline_result), "bm25", "bm25"))
        candidates.extend(candidate_rows(question, "bm25", "bm25", baseline_result))

        hybrid_result = hybrid.retrieve(question["question"], FINAL_TOP_K)
        rows.append(
            score_one(
                question,
                hit_pairs(hybrid_result),
                diagnostic_configuration["model"],
                diagnostic_configuration["mode"],
                diagnostic_configuration["bm25_weight"],
                diagnostic_configuration["dense_weight"],
            )
        )
        candidates.extend(
            candidate_rows(
                question,
                diagnostic_configuration["model"],
                diagnostic_configuration["mode"],
                hybrid_result,
            )
        )
    return rows, candidates


if RUN_EXPENSIVE:
    diagnostic_configuration = {
        "model": str(system_retriever["model"]),
        "mode": str(system_retriever["mode"]),
        "dense_weight": float(system_retriever["dense_weight"]),
        "bm25_weight": float(system_retriever["bm25_weight"]),
        "selected_on": "joint original and para-v1 273-question Dev comparison",
        "selection_basis": SYSTEM_SELECTION_BASIS,
        "selection_status": "selected_by_team_after_joint_robustness_review",
        "metric_winner_run_name": str(metric_winner["run_name"]),
        "holdout_used_for_selection": False,
    }
    holdout_rows, holdout_candidates = evaluate_diagnostic_holdout()
    assert len(holdout_rows) == len(heldout_gold) * 2
    assert {row["model"] for row in holdout_rows} == {
        "bm25",
        diagnostic_configuration["model"],
    }
    holdout_summary = sort_summary(summarize(holdout_rows))
    assert len(holdout_summary) == 2
    display(holdout_summary)

    m7_retrieval_rows = []
    m7_retrieval_path = OUTPUT_DIR / "m7_bge_m3_d0.10_b0.90_original_top20.jsonl"
    m7_manifest_path = OUTPUT_DIR / "m7_bge_m3_d0.10_b0.90_original_top20_manifest.json"
    if QUERY_SET == "original":
        m7_retriever, m7_index_artifact = build_project_retriever(
            SYSTEM_RETRIEVER_MODEL,
            RetrievalMode.HYBRID,
            dense_weight=0.10,
            bm25_weight=0.90,
        )
        for question in sorted(original_gold, key=lambda row: row["question_id"]):
            result = m7_retriever.retrieve(question["question"], RRF_INPUT_TOP_K)
            strict_retrieval = RetrievalResult.model_validate(result.model_dump(mode="json"))
            assert strict_retrieval.query == question["question"].strip()
            assert len(strict_retrieval.hits) == RRF_INPUT_TOP_K
            hit_metadata = []
            for hit in strict_retrieval.hits:
                chunk = by_chunk[hit.chunk_id]
                assert hit.text == chunk["text"]
                hit_metadata.append(
                    {
                        "rank": hit.rank,
                        "chunk_id": hit.chunk_id,
                        "provider": chunk["provider"],
                        "textbook_id": chunk["textbook_id"],
                        "document_id": chunk["document_id"],
                        "chapter_id": chunk["chapter_id"],
                        "source_name": chunk["source_name"],
                        "page_or_location": chunk["page_or_location"],
                        "source_locator": chunk["source_locator"],
                        "text": chunk["text"],
                    }
                )
            m7_retrieval_rows.append(
                {
                    "schema_version": "1.0",
                    "question_id": question["question_id"],
                    "execution_mode": "retrieval_only",
                    "status": "retrieved",
                    "retrieval": strict_retrieval.model_dump(mode="json"),
                    "hit_metadata": hit_metadata,
                }
            )
        assert len(m7_retrieval_rows) == 341
        assert len({row["question_id"] for row in m7_retrieval_rows}) == 341
        write_checkpoint(m7_retrieval_path, m7_retrieval_rows)
        m7_manifest = {
            "schema_version": "1.0",
            "result_type": "m6_system_retriever_candidates",
            "run_id": "m6-bge-m3-d0.10-b0.90-original-top20",
            "dataset_version": EVALUATION_SET_VERSION,
            "split": "all_diagnostic_341",
            "reportable": False,
            "execution_mode": "retrieval_only",
            "question_count": len(m7_retrieval_rows),
            "query_set": "original",
            "model": MODEL_SPECS[SYSTEM_RETRIEVER_MODEL]["model"],
            "model_revision": MODEL_SPECS[SYSTEM_RETRIEVER_MODEL]["revision"],
            "dense_weight": 0.10,
            "bm25_weight": 0.90,
            "rrf_k": RRF_K,
            "rrf_input_top_k": RRF_INPUT_TOP_K,
            "corpus_version": corpus_manifest["corpus_version"],
            "corpus_hash": corpus_manifest["corpus_hash"],
            "manifest_hash": corpus_manifest["manifest_hash"],
            "chunk_config_hash": corpus_manifest["chunk_config_hash"],
            "index_version": m7_index_artifact["artifact_id"],
            "gold_sha256": sha256(original_gold_bytes),
            "retrieval_runs_sha256": sha256(m7_retrieval_path.read_bytes()),
            "git_commit": run_git_state.commit,
        }
        write_utf8_lf(
            m7_manifest_path,
            json.dumps(m7_manifest, ensure_ascii=False, indent=2) + "\n",
        )
        print("M7 original-query Top-20 handoff:", m7_retrieval_path)
else:
    holdout_rows = holdout_candidates = []
    print("Diagnostic holdout run skipped. It is not a clean final Test set.")

Diagnostic holdout run skipped. It is not a clean final Test set.


## 13. Save auditable outputs and a clean-run manifest


In [14]:
def write_jsonl(path: Path, rows: list[dict]) -> None:
    write_utf8_lf(
        path,
        "".join(json.dumps(row, ensure_ascii=False, allow_nan=False) + "\n" for row in rows),
    )


if RUN_EXPENSIVE:
    final_git_state = capture_git_state(PROJECT_ROOT)
    assert final_git_state.commit == run_git_state.commit
    assert not final_git_state.dirty, "Repository changed while the experiment was running."

    split_assignments = [
        {
            "question_id": question["question_id"],
            "evaluation_split": question["evaluation_split"],
            "source_gold_split": question["source_gold_split"],
            "textbook_id": question_textbook_id(question),
            "question_type": question["question_type"],
            "question_difficulty": question["question_difficulty"],
        }
        for question in dev_gold + heldout_gold
    ]
    write_jsonl(OUTPUT_DIR / "evaluation_split_assignments.jsonl", split_assignments)
    write_jsonl(OUTPUT_DIR / "gold_span_bindings.jsonl", binding_rows)
    write_jsonl(OUTPUT_DIR / "dev_metrics.jsonl", dev_rows)
    write_jsonl(OUTPUT_DIR / "dev_candidate_hits.jsonl", dev_candidates)
    write_jsonl(OUTPUT_DIR / "diagnostic_holdout_metrics.jsonl", holdout_rows)
    write_jsonl(OUTPUT_DIR / "diagnostic_holdout_candidate_hits.jsonl", holdout_candidates)

    write_csv_lf(dev_summary, OUTPUT_DIR / "dev_summary.csv")
    write_csv_lf(dev_summary, OUTPUT_DIR / "dev_overall_ranked.csv")
    write_csv_lf(grouped_summary(dev_rows), OUTPUT_DIR / "dev_grouped_summary.csv")
    write_csv_lf(holdout_summary, OUTPUT_DIR / "diagnostic_holdout_summary.csv")

    result_files = [
        OUTPUT_DIR / "dev_summary.csv",
        OUTPUT_DIR / "dev_overall_ranked.csv",
        OUTPUT_DIR / "dev_grouped_summary.csv",
        OUTPUT_DIR / "dev_baseline_comparison.csv",
        OUTPUT_DIR / "diagnostic_holdout_summary.csv",
        OUTPUT_DIR / "truncation_diagnostics.csv",
        OUTPUT_DIR / "evaluation_split_assignments.jsonl",
        OUTPUT_DIR / "gold_span_bindings.jsonl",
    ]
    result_files.extend(
        path
        for path in (
            OUTPUT_DIR / "dev_rule1_selection.csv",
            OUTPUT_DIR / "dev_paired_bm25_vs_bge_m3_summary.csv",
            OUTPUT_DIR / "dev_paired_bm25_vs_bge_m3_questions.csv",
            OUTPUT_DIR / "m7_bge_m3_d0.10_b0.90_original_top20_manifest.json",
        )
        if path.is_file()
    )
    manifest = {
        "schema_version": "m6-w6-v2-four-book-project-retriever-3",
        "created_at_utc": datetime.now(UTC).isoformat(),
        "reportable": False,
        "evaluation_set_version": EVALUATION_SET_VERSION,
        "query_set": QUERY_SET,
        "gold_acceptance_policy": GOLD_ACCEPTANCE_POLICY,
        "source_annotation_status_counts": dict(
            collections.Counter(row["annotation_status"] for row in gold)
        ),
        "gold_annotation_version": GOLD_ANNOTATION_VERSION,
        "gold_row_count": len(gold),
        "gold_evidence_span_count": len(binding_rows),
        "gold_sha256": sha256(gold_bytes),
        "original_gold_sha256": sha256(original_gold_bytes),
        "github_gold_sha256": sha256(github_gold_bytes),
        "gold_source_corpus_versions": sorted(gold_source_corpus_versions),
        "gold_rebound_to_corpus_version": corpus_manifest["corpus_version"],
        "binding_sha256": sha256(binding_bytes),
        "git": {
            "commit": final_git_state.commit,
            "checkpoint_generation_commit": checkpoint_generation_commit,
            "dirty": final_git_state.dirty,
            "snapshot_sha256": final_git_state.snapshot_sha256,
        },
        "project_retriever": "cs30.pipeline.build_real_retrieval_deps",
        "method_config": {
            "rrf_k": RRF_K,
            "rrf_input_top_k": RRF_INPUT_TOP_K,
            "final_top_k": FINAL_TOP_K,
            "bm25_stopwords": True,
            "bm25_implementation": "cs30.retrieval.real.BM25Retriever",
            "rrf_implementation": "cs30.retrieval.real.RRFRetriever",
        },
        "corpus_version": corpus_manifest["corpus_version"],
        "corpus_hash": corpus_manifest["corpus_hash"],
        "corpus_manifest_hash": corpus_manifest["manifest_hash"],
        "chunk_config_hash": corpus_manifest["chunk_config_hash"],
        "required_textbook_ids": corpus_manifest["required_textbook_ids"],
        "included_textbook_ids": corpus_manifest["included_textbook_ids"],
        "corpus_zip_sha256": sha256(CORPUS_ZIP.read_bytes()),
        "m5_zip_sha256": sha256(M5_ZIP.read_bytes()),
        "models": {
            key: {
                "model": spec["model"],
                "revision": spec["revision"],
                "trust_remote_code": spec["trust_remote_code"],
                "query_instruction": spec["query_instruction"],
                "query_prompt_name": spec["query_prompt_name"],
                "index_sha256": index_provenance[key].get("metadata", {}).get("index_sha256"),
            }
            for key, spec in MODEL_SPECS.items()
        },
        "baseline_selection": json.loads(formal_baseline_selection.to_json()),
        "system_retriever_baseline_comparison": json.loads(system_baseline_comparison.to_json()),
        "dev_metric_winner": json.loads(metric_winner.to_json()),
        "system_retriever_selection": {
            "primary_metric": "combined original+para-v1 Dev Complete Hit@5",
            "tie_breaker": "question-count-weighted mean MRR",
            "joint_metric_winner_run_name": (
                str(joint_selection.iloc[0]["run_name"]) if "joint_selection" in globals() else None
            ),
            "selected_run_name": (f"{SYSTEM_RETRIEVER_MODEL}_{SYSTEM_RETRIEVER_MODE}"),
            "selected_joint_rank": (
                int(system_joint_row["rule1_rank"]) if "system_joint_row" in globals() else None
            ),
            "selection_type": "documented_team_operational_decision",
            "selection_basis": SYSTEM_SELECTION_BASIS,
            "decision_date": "2026-10-23",
            "statistical_superiority_claimed": False,
        },
        "diagnostic_configuration": diagnostic_configuration,
        "dev_question_count": len(dev_gold),
        "diagnostic_holdout_question_count": len(heldout_gold),
        "diagnostic_holdout_configuration_count": 2,
        "diagnostic_holdout_policy": (
            "BM25 and the BGE-M3 0.10/0.90 system retriever only; no all-model holdout sweep"
        ),
        "diagnostic_holdout_exposure_status": (
            "previously exposed to 64 configurations in a superseded run"
        ),
        "diagnostic_holdout_used_for_selection": False,
        "final_test_status": "not run; requires a new team-approved untouched split",
        "split_method": "deterministic textbook-quota evidence-component stratification",
        "split_seed": SPLIT_SEED,
        "split_stratify_fields": list(STRATIFY_FIELDS),
        "holdout_textbook_quotas": TEST_QUOTAS,
        "holdout_question_ids_sha256": TEST_QUESTION_IDS_SHA256,
        "cross_split_evidence_chunk_count": len(dev_evidence & test_evidence),
        "unanswerable_question_count": 0,
        "result_file_sha256": {path.name: sha256(path.read_bytes()) for path in result_files},
        "limitations": [
            "the 273/68 split is diagnostic and not team-approved as a final split",
            "the 68-question holdout was exposed to all configurations in a superseded run",
            "development corpus is reportable=false",
            "Gold annotation status is draft",
            "all questions are answerable",
        ],
    }
    write_utf8_lf(
        OUTPUT_DIR / "run_manifest.json",
        json.dumps(manifest, ensure_ascii=False, indent=2) + "\n",
    )
    print("Saved clean shared-retriever outputs:", OUTPUT_DIR)
else:
    print("Preflight only: no measured files were written.")

Preflight only: no measured files were written.

## 14. Published lightweight tables


In [15]:
KEY_METRIC_COLUMNS = [
    "run_name",
    "model",
    "mode",
    "dense_weight",
    "bm25_weight",
    "question_count",
    "complete_hit_at_5",
    "hit_at_5",
    "mrr",
    "complete_evidence_mrr",
    "hit_at_1",
    "hit_at_3",
    "hit_at_10",
    "complete_hit_at_1",
    "complete_hit_at_3",
    "complete_hit_at_10",
    "evidence_recall_at_1",
    "evidence_recall_at_3",
    "evidence_recall_at_5",
    "evidence_recall_at_10",
]


def display_complete_csv(path: Path, title: str) -> None:
    frame = pd.read_csv(path)
    leading = [column for column in KEY_METRIC_COLUMNS if column in frame.columns]
    remaining = [column for column in frame.columns if column not in leading]
    frame = frame.loc[:, leading + remaining]
    print(f"{title} ({len(frame)} rows x {len(frame.columns)} columns; all rows and columns shown)")
    with pd.option_context(
        "display.max_rows",
        None,
        "display.max_columns",
        None,
        "display.width",
        None,
        "display.max_colwidth",
        None,
    ):
        display(frame)


if result_path("dev_summary.csv").is_file():
    display_complete_csv(result_path("dev_summary.csv"), "Published Dev summary")
if result_path("dev_baseline_comparison.csv").is_file():
    display_complete_csv(
        result_path("dev_baseline_comparison.csv"),
        "Published Dev baseline decision",
    )
if result_path("diagnostic_holdout_summary.csv").is_file():
    display_complete_csv(
        result_path("diagnostic_holdout_summary.csv"),
        "Published exposed diagnostic holdout; not a clean final Test",
    )

Published Dev summary (64 rows x 81 columns; all rows and columns shown)


,run_name,model,mode,dense_weight,bm25_weight,question_count,complete_hit_at_5,hit_at_5,mrr,complete_evidence_mrr,hit_at_1,hit_at_3,hit_at_10,complete_hit_at_1,complete_hit_at_3,complete_hit_at_10,evidence_recall_at_1,evidence_recall_at_3,evidence_recall_at_5,evidence_recall_at_10,complete_evidence_hit_at_1,complete_evidence_hit_at_3,complete_evidence_hit_at_5,complete_evidence_hit_at_10,first_hit_mrr,recall_at_1,recall_at_3,recall_at_5,recall_at_10,core_precision_at_1,core_precision_at_3,core_precision_at_5,core_precision_at_10,context_noise_rate_at_1,context_noise_rate_at_3,context_noise_rate_at_5,context_noise_rate_at_10,no_relevant_hit_rate_at_1,no_relevant_hit_rate_at_3,no_relevant_hit_rate_at_5,no_relevant_hit_rate_at_10,partial_evidence_count_at_1,partial_evidence_count_at_3,partial_evidence_count_at_5,partial_evidence_count_at_10,returned_count_at_1,returned_count_at_3,returned_count_at_5,returned_count_at_10,empty_retrieval_rate_at_1,empty_retrieval_rate_at_3,empty_retrieval_rate_at_5,empty_retrieval_rate_at_10,no_hit_rate_at_1,no_hit_rate_at_3,no_hit_rate_at_5,no_hit_rate_at_10,returned_count,empty_retrieval,unanswerable_no_hit,unanswerable_false_positive,no_relevant_hit_at_1,no_relevant_hit_at_3,no_relevant_hit_at_5,no_relevant_hit_at_10,core_precision_at_1_defined_count,core_precision_at_3_defined_count,core_precision_at_5_defined_count,core_precision_at_10_defined_count,context_noise_rate_at_1_defined_count,context_noise_rate_at_3_defined_count,context_noise_rate_at_5_defined_count,context_noise_rate_at_10_defined_count,no_relevant_hit_rate_at_1_defined_count,no_relevant_hit_rate_at_3_defined_count,no_relevant_hit_rate_at_5_defined_count,no_relevant_hit_rate_at_10_defined_count,partial_evidence_count_at_1_defined_count,partial_evidence_count_at_3_defined_count,partial_evidence_count_at_5_defined_count,partial_evidence_count_at_10_defined_count
0,bge_base_hybrid_d0.05_b0.95,bge_base,hybrid_d0.05_b0.95,0.05,0.95,273,0.798535,0.802198,0.569639,0.567197,0.388278,0.743590,0.853480,0.388278,0.739927,0.846154,0.388278,0.741758,0.800366,0.849817,0.388278,0.739927,0.798535,0.846154,0.569639,0.388278,0.741758,0.800366,0.849817,0.388278,0.247863,0.160440,0.085348,0.611722,0.752137,0.839560,0.914652,0.611722,0.256410,0.197802,0.146520,0.0,0.0,0.0,0.0,1.0,3.0,5.0,10.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,10.0,0.0,NaN,NaN,0.611722,0.256410,0.197802,0.146520,273,273,273,273,273,273,273,273,273,273,273,273,273,273,273,273
1,bge_m3_hybrid_d0.10_b0.90,bge_m3,hybrid_d0.10_b0.90,0.10,0.90,273,0.794872,0.798535,0.545921,0.543566,0.358974,0.703297,0.853480,0.358974,0.699634,0.846154,0.358974,0.701465,0.796703,0.849817,0.358974,0.699634,0.794872,0.846154,0.545921,0.358974,0.701465,0.796703,0.849817,0.358974,0.234432,0.159707,0.085348,0.641026,0.765568,0.840293,0.914652,0.641026,0.296703,0.201465,0.146520,0.0,0.0,0.0,0.0,1.0,3.0,5.0,10.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,10.0,0.0,NaN,NaN,0.641026,0.296703,0.201465,0.146520,273,273,273,273,273,273,273,273,273,273,273,273,273,273,273,273
2,qwen3_hybrid_d0.05_b0.95,qwen3,hybrid_d0.05_b0.95,0.05,0.95,273,0.791209,0.798535,0.568835,0.566271,0.391941,0.736264,0.853480,0.391941,0.732601,0.846154,0.391941,0.734432,0.794872,0.849817,0.391941,0.732601,0.791209,0.846154,0.568835,0.391941,0.734432,0.794872,0.849817,0.391941,0.245421,0.159707,0.085348,0.608059,0.754579,0.840293,0.914652,0.608059,0.263736,0.201465,0.146520,0.0,0.0,0.0,0.0,1.0,3.0,5.0,10.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,10.0,0.0,NaN,NaN,0.608059,0.263736,0.201465,0.146520,273,273,273,273,273,273,273,273,273,273,273,273,273,273,273,273
3,qwen3_hybrid_d0.10_b0.90,qwen3,hybrid_d0.10_b0.90,0.10,0.90,273,0.791209,0.794872,0.556519,0.554164,0.373626,0.710623,0.860806,0.373626,0.706960,0.853480,0.373626,0.708791,0.793040,0.857143,0.373626,0.706960,0.791209,0.853480,0.556519,0.373626,0.708791,0.793040,0.857143,0.373626,0.236874,0.158974,0.086081,0.626374,0.763126,0.841026,0.913919,0.626374,0.289377,0.205128,0.139194,0.0,0.0,0.0,0.0,1.0,3.0,5.0,10.0,

Published Dev baseline decision (2 rows x 15 columns; all rows and columns shown)


,question_count,comparison_role,baseline_run_name,candidate_run_name,candidate_selection_basis,baseline_complete_hits_at_5,candidate_complete_hits_at_5,baseline_mrr,candidate_mrr,baseline_only_hits,candidate_only_hits,exact_mcnemar_p,passes_baseline_replacement_rule,selection_decision,selection_reason
0,273,dev_metric_winner,bm25_bm25,bge_base_hybrid_d0.05_b0.95,"highest complete_hit_at_5, then highest mrr, among Dev hybrids",215,218,0.575042,0.569639,0,3,0.250000,False,retain_bm25,candidate did not pass the preregistered Dev replacement rule
1,273,team_selected_provisional,bm25_bm25,bge_m3_hybrid_d0.10_b0.90,team-selected provisional retriever for downstream generator evaluation,215,217,0.575042,0.545921,4,6,0.753906,False,retain_bm25,candidate did not pass the preregistered Dev replacement rule


Published exposed diagnostic holdout; not a clean final Test (2 rows x 81 columns; all rows and columns shown)


,run_name,model,mode,dense_weight,bm25_weight,question_count,complete_hit_at_5,hit_at_5,mrr,complete_evidence_mrr,hit_at_1,hit_at_3,hit_at_10,complete_hit_at_1,complete_hit_at_3,complete_hit_at_10,evidence_recall_at_1,evidence_recall_at_3,evidence_recall_at_5,evidence_recall_at_10,complete_evidence_hit_at_1,complete_evidence_hit_at_3,complete_evidence_hit_at_5,complete_evidence_hit_at_10,first_hit_mrr,recall_at_1,recall_at_3,recall_at_5,recall_at_10,core_precision_at_1,core_precision_at_3,core_precision_at_5,core_precision_at_10,context_noise_rate_at_1,context_noise_rate_at_3,context_noise_rate_at_5,context_noise_rate_at_10,no_relevant_hit_rate_at_1,no_relevant_hit_rate_at_3,no_relevant_hit_rate_at_5,no_relevant_hit_rate_at_10,partial_evidence_count_at_1,partial_evidence_count_at_3,partial_evidence_count_at_5,partial_evidence_count_at_10,returned_count_at_1,returned_count_at_3,returned_count_at_5,returned_count_at_10,empty_retrieval_rate_at_1,empty_retrieval_rate_at_3,empty_retrieval_rate_at_5,empty_retrieval_rate_at_10,no_hit_rate_at_1,no_hit_rate_at_3,no_hit_rate_at_5,no_hit_rate_at_10,returned_count,empty_retrieval,unanswerable_no_hit,unanswerable_false_positive,no_relevant_hit_at_1,no_relevant_hit_at_3,no_relevant_hit_at_5,no_relevant_hit_at_10,core_precision_at_1_defined_count,core_precision_at_3_defined_count,core_precision_at_5_defined_count,core_precision_at_10_defined_count,context_noise_rate_at_1_defined_count,context_noise_rate_at_3_defined_count,context_noise_rate_at_5_defined_count,context_noise_rate_at_10_defined_count,no_relevant_hit_rate_at_1_defined_count,no_relevant_hit_rate_at_3_defined_count,no_relevant_hit_rate_at_5_defined_count,no_relevant_hit_rate_at_10_defined_count,partial_evidence_count_at_1_defined_count,partial_evidence_count_at_3_defined_count,partial_evidence_count_at_5_defined_count,partial_evidence_count_at_10_defined_count
0,bm25_bm25,bm25,bm25,NaN,NaN,68,0.750000,0.750000,0.484559,0.484559,0.279412,0.691176,0.764706,0.279412,0.691176,0.764706,0.279412,0.691176,0.750000,0.764706,0.279412,0.691176,0.750000,0.764706,0.484559,0.279412,0.691176,0.750000,0.764706,0.279412,0.230392,0.150000,0.076471,0.720588,0.769608,0.850000,0.923529,0.720588,0.308824,0.250000,0.235294,0.0,0.0,0.0,0.0,1.0,3.0,5.0,10.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,10.0,0.0,NaN,NaN,0.720588,0.308824,0.250000,0.235294,68,68,68,68,68,68,68,68,68,68,68,68,68,68,68,68
1,bge_m3_hybrid_d0.10_b0.90,bge_m3,hybrid_d0.10_b0.90,0.1,0.9,68,0.676471,0.691176,0.464496,0.460819,0.279412,0.632353,0.794118,0.279412,0.632353,0.779412,0.279412,0.632353,0.683824,0.786765,0.279412,0.632353,0.676471,0.779412,0.464496,0.279412,0.632353,0.683824,0.786765,0.279412,0.210784,0.138235,0.079412,0.720588,0.789216,0.861765,0.920588,0.720588,0.367647,0.308824,0.205882,0.0,0.0,0.0,0.0,1.0,3.0,5.0,10.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,10.0,0.0,NaN,NaN,0.720588,0.367647,0.308824,0.205882,68,68,68,68,68,68,68,68,68,68,68,68,68,68,68,68


## Interpretation guardrails

- Retrieval uses `cs30.pipeline.build_real_retrieval_deps`; the notebook does
  not maintain a second BM25 or RRF implementation.
- `rrf_input_top_k=20`, matching project configuration. Every model revision
  is a full 40-character commit hash; Jina and Nomic remote code is therefore
  pinned. Qwen3 uses `query_prompt_name="query"`.
- With `rrf_k=60` and `rrf_input_top_k=20`, every tested Hybrid having
  Dense weight at most 0.25 satisfies `dense_weight / 61 < bm25_weight / 80`.
  When BM25 returns 20 candidates, a Dense-only chunk therefore cannot enter
  the fused top 10; Dense mainly reranks BM25 candidates that also occur in
  the Dense top 20. This is a mechanism limitation, not evidence that Dense
  contributed new chunks to the reported top-10 results.
- Model and Hybrid-weight comparisons use only the 273-question diagnostic
  development split.
- The joint metric winner is reported separately from the system retriever.
  Qwen3 Dense 0.10 / BM25 0.90 ranks first by combined Complete Hit@5. BGE-M3
  Dense 0.10 / BM25 0.90 ranks second, one complete hit behind, and remains
  the team-selected M7 system retriever. No superiority claim is made.
- The 68-question holdout is already exposed and is not a clean final Test.
  A new team-approved untouched split is required for a final paper result.
- This four-book corpus is `reportable=false`; Gold annotations remain draft.
- All 341 questions are answerable, so unanswerable/refusal metrics are not
  supported by this dataset.
